In [1]:
### Load data
import pandas as pd   
# Prepare the historical panel (df1)
df1 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\B_Clustered_panel\03_Clustered_panel_1119.xlsx') 
df1 = df1.rename(columns={'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'})   # Rename the historical cooperation-mode column
df1.loc[df1['Both'] == 1, ['Conflict', 'Cooperation']] = 1
#df1['Cooperation'] = df1[label].any(axis=1).astype(int)   # Some panel rows have an active mode but Cooperation is not 1.
# Prepare future inputs (df2) and merge the mode indicators 
df2 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\E_ssp_predication\ssp1-5_future_data.xlsx', sheet_name='ssp5')  
df2 = df2[df2['year'] == 2050]  
# Prepare future cooperation-mode indicators (df3)
df3 = pd.read_csv(r'C:\Users\Lenovo\Desktop\RF_Water\D_strategy_onehot\20241218_updated_future_data_with_modes.csv')
df3.rename(columns={
    'ccode_iso3': 'country_code',
    'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'
}, inplace=True)   # Rename the country identifier and cooperation-mode column
df3 = df3[df3['year'] == 2050]   # Retain observations for 2050 only
# Define the selected columns
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative Planning and Adaptation Strategies'] 
# Select country_code and the mode-indicator columns from df3  
df3_labels = df3[['country_code'] + label]  
# Left-join df3_labels to df2  
df2 = pd.merge(df2, df3_labels, on='country_code', how='left')  
#df2.to_excel('D:\\OneDrive\\Desktop\\df2.xlsx', index=False)   # Optional intermediate output for inspection

In [2]:
### Combine historical and future inputs into df
import pandas as pd 
df2.rename(columns={  
    # Keep 'year' and 'export' unchanged when the column names already match 
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp5': 'population_density',  # For an SSP1 run, use the corresponding SSP1 field
    'GDP per capita_ssp5(constant 2015 US$)': 'gdp_percapita',  # For an SSP1 run, use the corresponding SSP1 field  
    'rural_population_ssp5(%)': 'rural_population',  # For an SSP1 run, use the corresponding SSP1 field
    'precipitation_ssp5': 'precipitation',   # For an SSP1 run, use the corresponding SSP1 field 
    'bws_ssp5': 'bws_score'}, inplace=True)   # For an SSP1 run, use the corresponding SSP1 field   
# Concatenate historical and future rows 
df = pd.concat([df1, df2], ignore_index=True, sort=True)
### Identify high-conflict countries for 2050 
df4 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\B_Clustered_panel\03_Clustered_panel_1119\Clustered_Conflict_Probability_241119.xlsx')
# Select rows assigned to Cluster 1  
high_conflict_country = df4[df4['Cluster'] == 1]  
# Extract the ISO3 country identifiers  
high_conflict_country = high_conflict_country['ccode_iso3'].tolist()  
### Set Conflict to 1 for high-conflict countries in 2050
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Conflict'] = 1
#df.to_excel('D:\\OneDrive\\Desktop\\df.xlsx', index=False)   # Optional intermediate output for inspection

In [3]:
'''
### Carry the 2019 Average_conflict value forward to 2050
# Select observations for 2019  
df_2019_Average_conflict = df[df['year'] == 2019][['ccode_iso3', 'Average_conflict']]  
# Create a dictionary of the 2019 country-level values  
conflict_dict = df_2019_Average_conflict.set_index('ccode_iso3')['Average_conflict'].to_dict()  
# Update the 2050 observations  
df.loc[df['year'] == 2050, 'Average_conflict'] = df[df['year'] == 2050]['ccode_iso3'].map(conflict_dict)  
'''
### Project Average_conflict to 2050 using a time-series model
df = df[df['ccode_iso3'] == 'LSO']
import pandas as pd  
from statsmodels.tsa.holtwinters import ExponentialSmoothing    
# Initialize a DataFrame for the 2050 forecasts  
predictions_2050 = pd.DataFrame(columns=['ccode_iso3', 'Average_conflict_2050'])  
# Forecast each country in high_conflict_country  
for country in high_conflict_country:  
    country_data = df[(df['ccode_iso3'] == country) & (df['year'].between(1951, 2019))]  
    if not country_data.empty:  
        print(f"Data for {country}:")  
        print(country_data[['Average_conflict']])  # Display the historical Average_conflict series for this country  
        # Fit exponential smoothing to the country-level time series  
        model = ExponentialSmoothing(country_data['Average_conflict'], trend='add', seasonal=None, seasonal_periods=1).fit()  
        # Forecast the series from 2020 through 2050  
        forecast = model.forecast(31)  # Forecast 31 steps beyond 2019 to reach 2050  
        print(f"Forecast for {country} from 2020 to 2050:")  
        print(forecast)  # Display the forecast  
        # Check that the forecast is available and whether its final value is negative  
        if not forecast.empty and forecast.iloc[-1] < 0:  
            forecast.iloc[-1] = 0  # Truncate a negative 2050 forecast to zero  
        # Check that a forecast is available after truncation  
        if not forecast.empty:  
            # Retain the forecast for 2050  
            forecast_2050 = forecast.iloc[-1]  
            new_prediction = pd.DataFrame({'ccode_iso3': [country], 'Average_conflict_2050': [forecast_2050]})  
            predictions_2050 = pd.concat([predictions_2050, new_prediction], ignore_index=True)  
        else:  
            print(f"No forecast available for {country}")  
# Merge the forecasts back into df  
df = df.merge(predictions_2050, on='ccode_iso3', how='left')  
# Update the projected values in the 2050 rows  
# Use the existing 2050 rows and update Average_conflict  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Average_conflict'] = df['Average_conflict_2050']  
# Remove the temporary forecast column  
df.drop(columns=['Average_conflict_2050'], inplace=True)  
# Optionally inspect the updated DataFrame  
# print(df)  
# Save the results to an Excel file  
df.to_excel('D:\\OneDrive\\Desktop\\time_series_LSO.xlsx', index=False)

Data for LSO:
      Average_conflict
6036          0.000000
6037          0.000000
6038          0.000000
6039          0.000000
6040          0.000000
6041          0.000000
6042          0.000000
6043          0.000000
6044          0.000000
6045          0.000000
6046          0.000000
6047          0.000000
6048          0.000000
6049          0.000000
6050          0.000000
6051          0.000000
6052          0.000000
6053          0.000000
6054          0.000000
6055          0.000000
6056          0.000000
6057          0.000000
6058          0.000000
6059          0.000000
6060          0.000000
6061          0.000000
6062          0.000000
6063          0.000000
6064          0.000000
6065          0.000000
...                ...
6075          0.050000
6076          0.048780
6077          0.047619
6078          0.046512
6079          0.045455
6080          0.066667
6081          0.065217
6082          0.106383
6083          0.104167
6084          0.102041
6085          0.1000

D:\Anaconda\lib\site-packages\statsmodels\tsa\base\tsa_model.py:221: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  ' ignored when e.g. forecasting.', ValueWarning)
D:\Anaconda\lib\site-packages\statsmodels\tsa\base\tsa_model.py:531: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  ValueWarning)
